In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib

df = pd.read_csv('../data/processed/churn_clean.csv')
print("Shape:", df.shape)
print(df.dtypes)

Shape: (5630, 19)
Churn                            int64
Tenure                         float64
PreferredLoginDevice            object
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode            object
Gender                          object
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferredOrderCat               object
SatisfactionScore                int64
MaritalStatus                   object
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                       int64
OrderCount                       int64
DaySinceLastOrder                int64
CashbackAmount                 float64
dtype: object


In [5]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(le_dict, '../models/label_encoders.pkl')
print("Encoders saved ✓")

Encoders saved ✓


In [7]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print("All folders ready ✓")

All folders ready ✓


In [9]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", categorical_cols)

le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le
    print(f"{col}: {list(le.classes_)}")

joblib.dump(le_dict, '../models/label_encoders.pkl')
print("\nEncoders saved ✓")

Categorical columns: []

Encoders saved ✓


In [11]:
# 1. Engagement Score — how active is this customer overall
df['EngagementScore'] = (
    df['HourSpendOnApp'] * 0.4 +
    df['OrderCount'] * 0.4 +
    df['CouponUsed'] * 0.2
)

# 2. Inactivity Flag — hasn't ordered in 7+ days
df['IsInactive'] = (df['DaySinceLastOrder'] >= 7).astype(int)

# 3. High Value Flag — earns above median cashback
df['IsHighValue'] = (df['CashbackAmount'] >= df['CashbackAmount'].median()).astype(int)

# 4. Complaint + Low Satisfaction — strongest combined churn signal
df['ComplainLowSat'] = ((df['Complain'] == 1) & (df['SatisfactionScore'] <= 2)).astype(int)

print("New features added:")
print(f"  EngagementScore — min: {df['EngagementScore'].min():.2f}, max: {df['EngagementScore'].max():.2f}")
print(f"  IsInactive      — {df['IsInactive'].sum()} customers flagged ({df['IsInactive'].mean()*100:.1f}%)")
print(f"  IsHighValue     — {df['IsHighValue'].sum()} customers flagged ({df['IsHighValue'].mean()*100:.1f}%)")
print(f"  ComplainLowSat  — {df['ComplainLowSat'].sum()} customers flagged ({df['ComplainLowSat'].mean()*100:.1f}%)")
print(f"\nNew shape: {df.shape}")

New features added:
  EngagementScore — min: 0.40, max: 10.80
  IsInactive      — 1749 customers flagged (31.1%)
  IsHighValue     — 2815 customers flagged (50.0%)
  ComplainLowSat  — 542 customers flagged (9.6%)

New shape: (5630, 23)


In [13]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nClass distribution before SMOTE:")
print(y.value_counts())
print(f"Churn rate: {y.mean():.2%}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")
print(f"Train churn rate: {y_train.mean():.2%}")
print(f"Test churn rate:  {y_test.mean():.2%}")

Features shape: (5630, 22)
Target shape: (5630,)

Class distribution before SMOTE:
Churn
0    4682
1     948
Name: count, dtype: int64
Churn rate: 16.84%

Train size: 4504
Test size:  1126
Train churn rate: 16.83%
Test churn rate:  16.87%


In [15]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(f"  Churned:  {(y_train == 1).sum()}")
print(f"  Retained: {(y_train == 0).sum()}")

print("\nAfter SMOTE:")
print(f"  Churned:  {(y_train_sm == 1).sum()}")
print(f"  Retained: {(y_train_sm == 0).sum()}")
print(f"  New train size: {len(y_train_sm)}")
print(f"  New churn rate: {y_train_sm.mean():.2%}")

Before SMOTE:
  Churned:  758
  Retained: 3746

After SMOTE:
  Churned:  3746
  Retained: 3746
  New train size: 7492
  New churn rate: 50.00%


In [17]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)

joblib.dump(scaler,               '../models/scaler.pkl')
joblib.dump(X.columns.tolist(),   '../models/feature_columns.pkl')

print("Saved:")
print("  scaler.pkl ✓")
print("  feature_columns.pkl ✓")
print(f"\nX_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

Saved:
  scaler.pkl ✓
  feature_columns.pkl ✓

X_train_scaled shape: (7492, 22)
X_test_scaled shape:  (1126, 22)
